In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
import numpy as np

In [4]:
# Install the Hugging Face datasets library (Run this cell once)
!pip install datasets -q

import pandas as pd
from datasets import load_dataset

print("Downloading and loading FreshRetailNet-50K from Hugging Face...")
ds = load_dataset("Dingdong-Inc/FreshRetailNet-50K")

# Convert to Pandas DataFrames
train_df = ds["train"].to_pandas()
eval_df = ds["eval"].to_pandas()

print("\nData successfully loaded!")
print("Train shape:", train_df.shape)
print("Eval shape :", eval_df.shape)

# Display the first 3 rows to confirm the data dictionary columns
print("\n--- Raw Train Data Preview ---")
display(train_df.head(3))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train.parquet:   0%|          | 0.00/106M [00:00<?, ?B/s]

data/eval.parquet:   0%|          | 0.00/8.44M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4500000 [00:00<?, ? examples/s]

Generating eval split:   0%|          | 0/350000 [00:00<?, ? examples/s]


Data successfully loaded!
Train shape: (4500000, 19)
Eval shape : (350000, 19)

--- Raw Train Data Preview ---


,city_id,store_id,management_group_id,first_category_id,second_category_id,third_category_id,product_id,dt,sale_amount,hours_sale,stock_hour6_22_cnt,hours_stock_status,discount,holiday_flag,activity_flag,precpt,avg_temperature,avg_humidity,avg_wind_level
0,0,0,0,5,6,65,38,2024-03-28,0.1,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.0, ...",0,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1.0,0,0,1.6999,15.48,73.54,1.97
1,0,0,0,5,6,65,38,2024-03-29,0.1,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.0, 0.0, ...",1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1.0,0,0,3.0190,15.08,76.56,1.71
2,0,0,0,5,6,65,38,2024-03-30,0.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1.0,1,0,2.0942,15.91,76.47,1.73


In [5]:
# ============================================================
# CELL 2A: SCHEMA AUDIT & DATA TYPES
# ============================================================
# WHY: Before any analysis, we must understand what each column
# represents, its dtype, and whether it needs transformation.
# This is especially critical here because two columns
# (hours_sale, hours_stock_status) are LIST columns — arrays
# of 24 hourly values stored per row. Mishandling them will
# silently break all downstream feature engineering.
#
# BUSINESS IMPACT: Knowing the schema prevents silent data
# corruption. A mislabeled category column treated as numeric
# would produce nonsense model weights, leading to bad orders.
# ============================================================



print("=" * 65)
print("  SCHEMA AUDIT — FreshRetailNet-50K")
print("=" * 65)

# Column descriptions (business glossary)
col_descriptions = {
    "city_id":              "City identifier (geographic market)",
    "store_id":             "Store identifier within a city",
    "management_group_id":  "Operational cluster / district group",
    "first_category_id":    "Top-level product category (e.g. Produce)",
    "second_category_id":   "Mid-level category (e.g. Vegetables)",
    "third_category_id":    "Sub-category (e.g. Leafy Greens)",
    "product_id":           "Unique SKU identifier",
    "dt":                   "Date of the record (daily grain)",
    "sale_amount":          "TARGET — Observed daily sales (kg/units)",
    "hours_sale":           "Array[24] — Hourly sales breakdown",
    "stock_hour6_22_cnt":   "# hours in-stock during 06:00–22:00",
    "hours_stock_status":   "Array[24] — 1=in-stock, 0=stockout per hour",
    "discount":             "Discount multiplier (1.0 = no discount)",
    "holiday_flag":         "1 if national/public holiday",
    "activity_flag":        "1 if a promotional activity is active",
    "precpt":               "Daily precipitation (mm)",
    "avg_temperature":      "Average daily temperature (°C)",
    "avg_humidity":         "Average daily humidity (%)",
    "avg_wind_level":       "Average wind level (Beaufort scale)",
}

schema_rows = []
for col in train_df.columns:
    sample_val = train_df[col].iloc[0]
    is_list_col = isinstance(sample_val, (list, np.ndarray))
    schema_rows.append({
        "Column":       col,
        "Dtype":        str(train_df[col].dtype),
        "Is_List":      is_list_col,
        "Null_Count":   train_df[col].isnull().sum(),
        "Null_%":       round(train_df[col].isnull().mean() * 100, 2),
        "Description":  col_descriptions.get(col, "—"),
    })

schema_df = pd.DataFrame(schema_rows)
display(schema_df.to_string(index=False))

list_cols = schema_df[schema_df["Is_List"]]["Column"].tolist()
scalar_cols = schema_df[~schema_df["Is_List"]]["Column"].tolist()

print(f"\n List (array) columns   : {list_cols}")
print(f"\n Scalar columns ({len(scalar_cols)})    : {scalar_cols}")
print(f"\nSchema audit complete. No structural surprises = safe to proceed.")

  SCHEMA AUDIT — FreshRetailNet-50K


'             Column   Dtype  Is_List  Null_Count  Null_%                                 Description\n            city_id   int64    False           0     0.0         City identifier (geographic market)\n           store_id   int64    False           0     0.0              Store identifier within a city\nmanagement_group_id   int64    False           0     0.0        Operational cluster / district group\n  first_category_id   int64    False           0     0.0   Top-level product category (e.g. Produce)\n second_category_id   int64    False           0     0.0        Mid-level category (e.g. Vegetables)\n  third_category_id   int64    False           0     0.0            Sub-category (e.g. Leafy Greens)\n         product_id   int64    False           0     0.0                       Unique SKU identifier\n                 dt  object    False           0     0.0            Date of the record (daily grain)\n        sale_amount float64    False           0     0.0    TARGET — Observed dai


 List (array) columns   : ['hours_sale', 'hours_stock_status']

 Scalar columns (17)    : ['city_id', 'store_id', 'management_group_id', 'first_category_id', 'second_category_id', 'third_category_id', 'product_id', 'dt', 'sale_amount', 'stock_hour6_22_cnt', 'discount', 'holiday_flag', 'activity_flag', 'precpt', 'avg_temperature', 'avg_humidity', 'avg_wind_level']

Schema audit complete. No structural surprises = safe to proceed.


In [6]:
# ============================================================
# CELL 2B: DATE RANGE & CONTINUITY CHECK
# ============================================================
# WHY: This is a TIME-SERIES problem. Gaps in dates would
# invalidate lag features and SARIMA/Prophet models.
# We need to confirm the 90-day contiguous window and locate
# where train ends and the final holdout (eval_df) begins.
#
# BUSINESS IMPACT: A missing week of data in a seasonal period
# (e.g. Chinese New Year) would cause the model to never learn
# that demand spike — leading to catastrophic under-ordering.
# ============================================================

train_df["dt"] = pd.to_datetime(train_df["dt"])
eval_df["dt"]  = pd.to_datetime(eval_df["dt"])

train_dates = train_df["dt"].sort_values().unique()
eval_dates  = eval_df["dt"].sort_values().unique()

print("=" * 55)
print("  DATE RANGE SUMMARY")
print("=" * 55)
print(f"  Train  : {train_dates.min().date()}  →  {train_dates.max().date()}")
print(f"  Eval   : {eval_dates.min().date()}   →  {eval_dates.max().date()}")
print(f"  Train days : {len(train_dates)}")
print(f"  Eval  days : {len(eval_dates)}")

# Check for gaps
all_expected = pd.date_range(train_dates.min(), train_dates.max(), freq="D")
missing_days = set(all_expected) - set(train_dates)
if missing_days:
    print(f"\n⚠️  GAPS DETECTED in train: {sorted(missing_days)}")
else:
    print(f"\n✅ No date gaps in train data — fully contiguous.")

# Check for overlap between train and eval
overlap = set(pd.DatetimeIndex(train_dates)) & set(pd.DatetimeIndex(eval_dates))
if overlap:
    print(f"⚠️  OVERLAP between train and eval: {sorted(overlap)}")
else:
    print(f"✅ No date overlap between train and eval — clean holdout.")

  DATE RANGE SUMMARY
  Train  : 2024-03-28  →  2024-06-25
  Eval   : 2024-06-26   →  2024-07-02
  Train days : 90
  Eval  days : 7

✅ No date gaps in train data — fully contiguous.
✅ No date overlap between train and eval — clean holdout.


In [7]:
# ============================================================
# CELL 2C: TEMPORAL TRAIN / VALIDATION SPLIT
# ============================================================
# WHY: For time-series data, we NEVER use random splits.
# A random split allows the model to "see the future" during
# training (data leakage), which causes wildly optimistic
# offline metrics that collapse in production.
#
# STRATEGY:
#   train_df  (4.5M rows, ~90 days)
#     └── train_split_df  → first 80% of days  (model training)
#     └── val_split_df    → last  20% of days  (offline validation)
#   eval_df   (350K rows) → FINAL HOLDOUT — not touched until
#                           the very last step of the project.
#
# 80/20 RATIONALE: With 90 days of data, 20% ≈ 18 days which
# is long enough to capture two full weekly cycles — the minimum
# needed to validate weekly seasonality (RQ-a).
#
# BUSINESS IMPACT: A clean temporal holdout simulates real
# deployment: the model trains on historical data and is judged
# on future unseen demand — exactly how it would run in prod.
# ============================================================

sorted_dates = sorted(train_df["dt"].unique())
n_days       = len(sorted_dates)
split_idx    = int(n_days * 0.80)

train_cutoff = sorted_dates[split_idx - 1]   # last day of training
val_start    = sorted_dates[split_idx]        # first day of validation

train_split_df = train_df[train_df["dt"] <= train_cutoff].copy()
val_split_df   = train_df[train_df["dt"] >  train_cutoff].copy()

print("=" * 60)
print("  TEMPORAL SPLIT SUMMARY")
print("=" * 60)
print(f"  Total unique days in train_df : {n_days}")
print(f"  Training window   : {sorted_dates[0].date()}  →  {train_cutoff.date()}  ({split_idx} days)")
print(f"  Validation window : {val_start.date()}        →  {sorted_dates[-1].date()}  ({n_days - split_idx} days)")
print(f"\n  train_split_df  rows : {len(train_split_df):,}")
print(f"  val_split_df    rows : {len(val_split_df):,}")
print(f"  eval_df (holdout)    : {len(eval_df):,}  ← DO NOT TOUCH until final evaluation")

# Sanity check: no leakage
assert train_split_df["dt"].max() < val_split_df["dt"].min(), "❌ LEAKAGE DETECTED"
assert val_split_df["dt"].max()   < eval_df["dt"].min(),      "❌ LEAKAGE: val overlaps holdout"
print("\n✅ All split assertions passed. Zero leakage confirmed.")

  TEMPORAL SPLIT SUMMARY
  Total unique days in train_df : 90
  Training window   : 2024-03-28  →  2024-06-07  (72 days)
  Validation window : 2024-06-08        →  2024-06-25  (18 days)

  train_split_df  rows : 3,600,000
  val_split_df    rows : 900,000
  eval_df (holdout)    : 350,000  ← DO NOT TOUCH until final evaluation

✅ All split assertions passed. Zero leakage confirmed.


In [8]:
# ============================================================
# CELL 2D: SCALE & CARDINALITY CHECK
# ============================================================
# WHY: Knowing unique counts for IDs tells us the complexity
# of the forecasting problem and whether we need product-level
# or store-level models. High cardinality drives the decision
# to use segmented models vs. a single global model.
#
# BUSINESS IMPACT: A retailer with 500 SKUs × 50 stores needs
# 25,000 micro-models — computationally infeasible. Cardinality
# analysis justifies the SEGMENTED strategy (Short/Medium/Long
# shelf life) that reduces model count while preserving accuracy.
# ============================================================

id_cols = ["city_id", "store_id", "management_group_id",
           "first_category_id", "second_category_id",
           "third_category_id", "product_id"]

print("=" * 55)
print("  CARDINALITY & SCALE (train_split_df)")
print("=" * 55)
for col in id_cols:
    n = train_split_df[col].nunique()
    print(f"  {col:<25} : {n:>6,} unique values")

print(f"\n  Total SKU × Store combinations : "
      f"{train_split_df.groupby(['store_id','product_id']).ngroups:,}")

print("\n--- Target Variable (sale_amount) Distribution ---")
desc = train_split_df["sale_amount"].describe(percentiles=[.25,.50,.75,.90,.99])
print(desc.round(4).to_string())

zero_pct = (train_split_df["sale_amount"] == 0).mean() * 100
print(f"\n  Zero-sale rows : {zero_pct:.1f}%")
print("\n  ⚠️  High zero % is expected for perishables — products")
print("  are not ordered every day. This drives the need for a")
print("  stockout-aware demand model (RQ-b).")

  CARDINALITY & SCALE (train_split_df)
  city_id                   :     18 unique values
  store_id                  :    898 unique values
  management_group_id       :      7 unique values
  first_category_id         :     32 unique values
  second_category_id        :     84 unique values
  third_category_id         :    233 unique values
  product_id                :    865 unique values

  Total SKU × Store combinations : 50,000

--- Target Variable (sale_amount) Distribution ---
count    3.600000e+06
mean     9.653000e-01
std      1.313200e+00
min      0.000000e+00
25%      4.000000e-01
50%      6.000000e-01
75%      1.100000e+00
90%      1.900000e+00
99%      5.600000e+00
max      3.610000e+01

  Zero-sale rows : 4.7%

  ⚠️  High zero % is expected for perishables — products
  are not ordered every day. This drives the need for a
  stockout-aware demand model (RQ-b).


In [9]:
# ============================================================
# CELL 3A: VERIFY STOCK STATUS ENCODING
# ============================================================
# WHY: Our censoring logic assumed 0=stockout, 1=in-stock.
# But the results show the opposite business logic:
#   - "Censored" days have HIGHER sales (1.012 vs 0.114)
#   - This means the product was IN STOCK on those days
# We need to read the raw array and confirm the encoding,
# then fix the is_censored_day flag before EDA.
#
# BUSINESS IMPACT: An inverted stockout flag would cause the
# model to LEARN FROM the wrong days — training on "normal"
# days that are actually stockouts and vice versa. Every
# downstream insight (RQ-b, RQ-c) would be corrupted.
# ============================================================

import numpy as np

# Step 1: Inspect a known-stockout row (sale_amount = 0)
print("=== INSPECTING ZERO-SALE ROWS (Expected: out of stock) ===\n")
zero_rows = train_split_df[train_split_df["sale_amount"] == 0].head(3)
for idx, row in zero_rows.iterrows():
    arr = np.array(row["hours_stock_status"])
    print(f"Row {idx} | sale_amount={row['sale_amount']} | stock_hour6_22_cnt={row['stock_hour6_22_cnt']}")
    print(f"  hours_stock_status (full 24h): {arr.tolist()}")
    print(f"  Business hours [6-21] values : {arr[6:22].tolist()}")
    print(f"  Count of 1s in business hrs  : {arr[6:22].sum()}")
    print(f"  Count of 0s in business hrs  : {(arr[6:22]==0).sum()}")
    print()

# Step 2: Inspect a high-sale row (Expected: in stock all day)
print("=== INSPECTING HIGH-SALE ROWS (Expected: well stocked) ===\n")
high_rows = train_split_df[train_split_df["sale_amount"] > 3].head(3)
for idx, row in high_rows.iterrows():
    arr = np.array(row["hours_stock_status"])
    print(f"Row {idx} | sale_amount={row['sale_amount']} | stock_hour6_22_cnt={row['stock_hour6_22_cnt']}")
    print(f"  hours_stock_status (full 24h): {arr.tolist()}")
    print(f"  Business hours [6-21] values : {arr[6:22].tolist()}")
    print(f"  Count of 1s in business hrs  : {arr[6:22].sum()}")
    print(f"  Count of 0s in business hrs  : {(arr[6:22]==0).sum()}")
    print()

# Step 3: Cross-check with the named column stock_hour6_22_cnt
# Dataset says this = "# hours in-stock during 06:00–22:00"
# If stock_hour6_22_cnt is HIGH on high-sale days, then 1=in-stock
print("=== CROSS-CHECK: stock_hour6_22_cnt vs sale_amount ===")
corr = train_split_df[["sale_amount", "stock_hour6_22_cnt"]].corr()
print(corr)
print("\nIf correlation is POSITIVE → 1=in-stock (our original assumption was WRONG)")
print("If correlation is NEGATIVE → 1=stockout (our original assumption was correct)")

=== INSPECTING ZERO-SALE ROWS (Expected: out of stock) ===

Row 2 | sale_amount=0.0 | stock_hour6_22_cnt=0
  hours_stock_status (full 24h): [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
  Business hours [6-21] values : [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
  Count of 1s in business hrs  : 0
  Count of 0s in business hrs  : 16

Row 8 | sale_amount=0.0 | stock_hour6_22_cnt=0
  hours_stock_status (full 24h): [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
  Business hours [6-21] values : [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
  Count of 1s in business hrs  : 0
  Count of 0s in business hrs  : 16

Row 10 | sale_amount=0.0 | stock_hour6_22_cnt=0
  hours_stock_status (full 24h): [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
  Business hours [6-21] values : [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
  Count of 1s in business hrs  : 0
  Count of 0s in business hrs  : 16

=== INSPECTING HIG

In [11]:
# ============================================================
# CELL 3B: FINAL CORRECTED CENSORING LOGIC
# ============================================================
# CONFIRMED ENCODING:
#   hours_stock_status: 1 = OUT-OF-STOCK, 0 = IN-STOCK
#   stock_hour6_22_cnt = count of 1s in business hours
#                      = number of STOCKOUT hours (6–22)
#
# TWO DISTINCT ZERO-SALE SCENARIOS (critical distinction):
#
#   TYPE A — "Not Stocked" day:
#     sale_amount = 0, stock_hour6_22_cnt = 0, all zeros in array
#     → Product was never put on shelf that day
#     → NOT a censored demand signal — true demand was zero
#     → Common for slow-moving SKUs on low-traffic days
#
#   TYPE B — "Sold Out" day (CENSORED):
#     sale_amount > 0 (or = 0 if stocked out from opening),
#     stock_hour6_22_cnt > 0, 1s appear in array
#     → Product WAS stocked but demand exceeded supply
#     → Observed sales UNDERESTIMATE true demand
#     → THIS is the censoring problem we need to correct
#
# BUSINESS IMPACT OF DISTINGUISHING THESE:
#   If we treat Type A as censored, we over-order products
#   that genuinely have zero demand → direct spoilage cost.
#   If we miss Type B censoring, we under-order hot products
#   → lost revenue on highest-margin items.
# ============================================================

import numpy as np

# Use stock_hour6_22_cnt directly (pre-computed, reliable)
# It counts the exact number of 1s (stockout hours) in [6–22]

for df in [train_split_df, val_split_df, eval_df]:

    # Stockout hours = directly from named column
    df["stockout_hours"] = df["stock_hour6_22_cnt"].astype(int)

    # TYPE A: Never stocked (no stockout hours AND zero sales)
    df["is_not_stocked"] = (
        (df["stockout_hours"] == 0) & (df["sale_amount"] == 0)
    ).astype(int)

    # TYPE B: Sold out mid-day (censored demand)
    # Threshold: stockout for 3+ hours = lost ~19%+ of selling window
    df["is_censored_day"] = (df["stockout_hours"] >= 3).astype(int)

    # Partial stockout flag (1–2 hours): minor impact, tracked separately
    df["is_partial_stockout"] = (
        (df["stockout_hours"] >= 1) & (df["stockout_hours"] < 3)
    ).astype(int)

# ── Validation Report ──────────────────────────────────────
print("=" * 60)
print("  CENSORING ANALYSIS REPORT (train_split_df)")
print("=" * 60)

total = len(train_split_df)

type_a  = train_split_df["is_not_stocked"].sum()
type_b  = train_split_df["is_censored_day"].sum()
partial = train_split_df["is_partial_stockout"].sum()
normal  = total - type_a - type_b - partial

print(f"\n  {'Category':<30} {'Rows':>10}  {'%':>7}  {'Avg Sales':>10}")
print(f"  {'-'*60}")

for label, mask in [
    ("Normal (fully in-stock)",   train_split_df["is_censored_day"]==0),
    ("TYPE B: Sold-out/Censored", train_split_df["is_censored_day"]==1),
    ("TYPE A: Never stocked",     train_split_df["is_not_stocked"]==1),
    ("Partial stockout (1-2 hr)", train_split_df["is_partial_stockout"]==1),
]:
    sub      = train_split_df[mask]
    avg_sale = sub["sale_amount"].mean()
    print(f"  {label:<30} {len(sub):>10,}  {len(sub)/total*100:>6.1f}%  {avg_sale:>10.3f}")

print(f"\n  {'TOTAL':<30} {total:>10,}  {'100.0%':>7}")

# Key business insight
censored_avg = train_split_df[train_split_df["is_censored_day"]==1]["sale_amount"].mean()
normal_avg   = train_split_df[
    (train_split_df["is_censored_day"]==0) &
    (train_split_df["is_not_stocked"]==0)
]["sale_amount"].mean()

demand_suppression = (normal_avg - censored_avg) / normal_avg * 100

print(f"\n  📊 KEY INSIGHT:")
print(f"  In-stock day avg sales   : {normal_avg:.3f}")
print(f"  Censored day avg sales   : {censored_avg:.3f}")
print(f"  Observed demand suppression due to stockouts: {demand_suppression:.1f}%")
print(f"\n  This means a naive model trained on raw sales will")
print(f"  UNDERESTIMATE true demand by ~{demand_suppression:.0f}% on censored days.")
print(f"  Stockout-aware correction directly addresses RQ-b and RQ-c.")

  CENSORING ANALYSIS REPORT (train_split_df)

  Category                             Rows        %   Avg Sales
  ------------------------------------------------------------
  Normal (fully in-stock)         2,208,262    61.3%       0.980
  TYPE B: Sold-out/Censored       1,391,738    38.7%       0.942
  TYPE A: Never stocked              37,956     1.1%       0.000
  Partial stockout (1-2 hr)         218,981     6.1%       1.336

  TOTAL                           3,600,000   100.0%

  📊 KEY INSIGHT:
  In-stock day avg sales   : 0.997
  Censored day avg sales   : 0.942
  Observed demand suppression due to stockouts: 5.5%

  This means a naive model trained on raw sales will
  UNDERESTIMATE true demand by ~5% on censored days.
  Stockout-aware correction directly addresses RQ-b and RQ-c.


In [12]:
# ============================================================
# CELL 3C: SHELF LIFE SEGMENTATION
# ============================================================
# WHY: 865 SKUs × 898 stores = 776,570 possible series.
# Building individual models is infeasible. We need segments
# that group products with similar spoilage risk profiles.
#
# Since the dataset does not provide explicit shelf life values,
# we use third_category_id as a proxy — products in the same
# sub-category share similar handling and shelf life properties.
#
# SEGMENT DEFINITIONS (to be validated against category map):
#   Short_Life  : third_category_id ≤ 80   (~1–3 days)
#   Medium_Life : third_category_id 81–160 (~4–7 days)
#   Long_Life   : third_category_id > 160  (~8+ days)
#
# Each segment gets its own model strategy:
#   Short_Life  → Random Forest Only (lower variance needed)
#   Medium_Life → Ensemble RF + LightGBM 50/50
#   Long_Life   → Ensemble RF + LightGBM 50/50
# ============================================================

def assign_shelf_life(cat_id):
    if cat_id <= 80:
        return "Short_Life"
    elif cat_id <= 160:
        return "Medium_Life"
    else:
        return "Long_Life"

for df in [train_split_df, val_split_df, eval_df]:
    df["shelf_life_bucket"] = df["third_category_id"].apply(assign_shelf_life)

print("=" * 65)
print("  SHELF LIFE SEGMENTATION REPORT (train_split_df)")
print("=" * 65)
print(f"\n  {'Segment':<15} {'Rows':>10} {'%':>7} {'SKUs':>7} "
      f"{'Avg Sale':>10} {'Censored%':>11}")
print(f"  {'-'*63}")

for bucket in ["Short_Life", "Medium_Life", "Long_Life"]:
    sub       = train_split_df[train_split_df["shelf_life_bucket"] == bucket]
    n_skus    = sub["product_id"].nunique()
    avg_sale  = sub["sale_amount"].mean()
    cens_pct  = sub["is_censored_day"].mean() * 100
    print(f"  {bucket:<15} {len(sub):>10,} {len(sub)/len(train_split_df)*100:>6.1f}% "
          f"{n_skus:>7,} {avg_sale:>10.3f} {cens_pct:>10.1f}%")

print(f"\n  {'TOTAL':<15} {len(train_split_df):>10,} {'100.0%':>7} "
      f"{train_split_df['product_id'].nunique():>7,}")
print("\n✅ Segmentation complete.")

  SHELF LIFE SEGMENTATION REPORT (train_split_df)

  Segment               Rows       %    SKUs   Avg Sale   Censored%
  ---------------------------------------------------------------
  Short_Life         940,104   26.1%     302      1.110       37.9%
  Medium_Life      1,708,128   47.4%     348      0.909       40.0%
  Long_Life          951,768   26.4%     215      0.924       37.1%

  TOTAL            3,600,000  100.0%     865

✅ Segmentation complete.


In [13]:
# ============================================================
# CELL 3D: SAVE FINAL SPLITS
# ============================================================

SAVE_DIR = "/content/drive/MyDrive/Project/Capstone/Data/Processed/"

train_split_df.to_parquet(SAVE_DIR + "train_split.parquet", index=False)
val_split_df.to_parquet(SAVE_DIR   + "val_split.parquet",   index=False)
eval_df.to_parquet(SAVE_DIR        + "eval_holdout.parquet", index=False)

print("✅ Final splits saved with all engineered columns:")
print("   + stockout_hours      (= stock_hour6_22_cnt, 1=stockout confirmed)")
print("   + is_not_stocked      (Type A zero: never put on shelf)")
print("   + is_censored_day     (Type B: sold out mid-day, demand suppressed)")
print("   + is_partial_stockout (1–2 hr minor stockout)")
print("   + shelf_life_bucket   (Short / Medium / Long)")
print(f"\n   train_split_df : {len(train_split_df):,} rows  (2024-03-28 → 2024-06-07)")
print(f"   val_split_df   : {len(val_split_df):,} rows  (2024-06-08 → 2024-06-25)")
print(f"   eval_df        : {len(eval_df):,} rows  (2024-06-26 → 2024-07-02) ← HOLDOUT")
print("\n🚀 Data preparation 100% complete. Ready for 01_EDA.ipynb")

✅ Final splits saved with all engineered columns:
   + stockout_hours      (= stock_hour6_22_cnt, 1=stockout confirmed)
   + is_not_stocked      (Type A zero: never put on shelf)
   + is_censored_day     (Type B: sold out mid-day, demand suppressed)
   + is_partial_stockout (1–2 hr minor stockout)
   + shelf_life_bucket   (Short / Medium / Long)

   train_split_df : 3,600,000 rows  (2024-03-28 → 2024-06-07)
   val_split_df   : 900,000 rows  (2024-06-08 → 2024-06-25)
   eval_df        : 350,000 rows  (2024-06-26 → 2024-07-02) ← HOLDOUT

🚀 Data preparation 100% complete. Ready for 01_EDA.ipynb
